In [1]:
from pyspark.sql import functions as F

air = spark.table("silver_mesures_air")
met = spark.table("silver_mesures_meteo")

# --- Gold 1 : agrégats journaliers par ville ---
gold_resume = (air
    .filter(F.col("qualite") == "OK")
    .withColumn("date_mesure", F.to_date("horodatage_collecte"))
    .groupBy("ville", "date_mesure")
    .agg(
        F.round(F.avg("aqi"), 2).alias("aqi_moyen"),
        F.min("aqi").alias("aqi_min"),
        F.max("aqi").alias("aqi_max"),
        F.count("*").alias("nb_mesures"),
        F.round(F.avg("pm25_idx"), 2).alias("pm25_moyen"),
        F.round(F.avg("pm10_idx"), 2).alias("pm10_moyen"),
    )
    .withColumn("niveau",
        F.when(F.col("aqi_moyen") <= 50, "Bon")
         .when(F.col("aqi_moyen") <= 100, "Modéré")
         .when(F.col("aqi_moyen") <= 150, "Mauvais pour les sensibles")
         .otherwise("Mauvais"))
)

gold_resume.write.format("delta").mode("overwrite").saveAsTable("gold_resume_journalier")
print(f"gold_resume_journalier : {gold_resume.count()} lignes")
gold_resume.orderBy("ville", "date_mesure").show(truncate=False)

StatementMeta(, 5aa59cd5-18c7-4429-8a36-e63b30503854, 3, Finished, Available, Finished, True)

gold_resume_journalier : 19 lignes
+-----------+-----------+---------+-------+-------+----------+----------+----------+------+
|ville      |date_mesure|aqi_moyen|aqi_min|aqi_max|nb_mesures|pm25_moyen|pm10_moyen|niveau|
+-----------+-----------+---------+-------+-------+----------+----------+----------+------+
|Bordeaux   |2026-09-02 |31.0     |30.0   |32.0   |2         |25.5      |15.5      |Bon   |
|Bordeaux   |2026-09-14 |52.0     |52.0   |52.0   |1         |21.0      |30.0      |Modéré|
|Lille      |2026-09-02 |61.0     |61.0   |61.0   |2         |61.0      |8.0       |Modéré|
|Lille      |2026-09-14 |61.0     |61.0   |61.0   |1         |61.0      |8.0       |Modéré|
|Lyon       |2026-09-02 |30.0     |30.0   |30.0   |1         |30.0      |29.0      |Bon   |
|Marseille  |2026-09-02 |42.5     |35.0   |50.0   |2         |32.0      |23.0      |Bon   |
|Marseille  |2026-09-14 |51.0     |51.0   |51.0   |1         |28.0      |12.0      |Modéré|
|Montpellier|2026-09-02 |44.5     |42.0   |47

In [2]:
# --- Gold 2 : dataset ML (jointure air + météo) ---
gold_dataset = (air.alias("a")
    .join(met.alias("m"), ["ville", "horodatage_collecte"], "inner")
    .filter((F.col("a.qualite") == "OK") & (F.col("m.qualite") == "OK"))
    .select(
        "ville",
        "horodatage_collecte",
        F.col("a.aqi").alias("aqi"),
        F.col("m.temperature"),
        F.col("m.humidite"),
        F.col("m.pression"),
        F.col("m.vitesse_vent"),
        F.col("m.precipitation"),
        F.col("m.nuages_pct"),
        F.hour("horodatage_collecte").alias("heure"),
        F.month("horodatage_collecte").alias("mois"),
        F.dayofweek("horodatage_collecte").alias("jour_semaine"),
    )
)

gold_dataset.write.format("delta").mode("overwrite").saveAsTable("gold_dataset_ml")
print(f"gold_dataset_ml : {gold_dataset.count()} lignes")
gold_dataset.show(5, truncate=False)

StatementMeta(, 5aa59cd5-18c7-4429-8a36-e63b30503854, 4, Finished, Available, Finished, False)

gold_dataset_ml : 28 lignes
+--------+-------------------+----+-----------+--------+--------+------------+-------------+----------+-----+----+------------+
|ville   |horodatage_collecte|aqi |temperature|humidite|pression|vitesse_vent|precipitation|nuages_pct|heure|mois|jour_semaine|
+--------+-------------------+----+-----------+--------+--------+------------+-------------+----------+-----+----+------------+
|Bordeaux|2026-09-02 10:00:00|30.0|25.61      |50.0    |1024.0  |3.09        |0.0          |84.0      |10   |9   |4           |
|Bordeaux|2026-09-02 13:00:00|32.0|28.74      |43.0    |1022.0  |4.12        |0.0          |88.0      |13   |9   |4           |
|Bordeaux|2026-09-14 12:00:00|52.0|32.15      |23.0    |1022.0  |1.03        |0.0          |6.0       |12   |9   |2           |
|Lille   |2026-09-02 10:00:00|61.0|20.36      |69.0    |1021.0  |4.63        |0.0          |79.0      |10   |9   |4           |
|Lille   |2026-09-02 13:00:00|61.0|22.14      |61.0    |1021.0  |5.14       